
<a id="top"></a>
# Modulo 4 — Marketing bancario, pipeline e tracciamento con MLflow

## Indice

- [Contesto: perché parlare di MLOps](#contesto-mlops)
- [Setup: librerie, esperimento MLflow e dati grezzi](#setup)
- [Preparazione del dataset e train/test split](#prep-split)
- [Pipeline di feature engineering con feature-engine](#pipeline-feature-engine)
- [Addestramento e confronto di più modelli](#confronto-modelli)
- [Lettura degli esperimenti dalla UI di MLflow](#mlflow-ui)
- [Esercizio guidato](#esercizio)
- [Dal modello tracciato all’inferenza riutilizzabile](#modello-inferenza)



<a id="contesto-mlops"></a>
## Contesto: perché parlare di MLOps

Finora il lavoro è rimasto dentro al notebook: lettura dei dati, preparazione, scelta di un modello e prima validazione.  
Funziona, ma è fragile. Piccole differenze di ambiente, trasformazioni fatte a mano e modelli non tracciati rendono difficile riprodurre i risultati o consegnare il lavoro a un team di ingegneri.

Qui entra in gioco il punto di vista **MLOps**. L'idea è trattare il ciclo di vita del modello in modo più simile al software:

- passaggi chiari e ripetibili;
- esperimenti tracciati;
- modelli salvati in un formato standard e riutilizzabile.

In questo notebook useremo **MLflow** per tracciare gli esperimenti e lo standardizzare il salvataggio dei modelli, e **feature-engine** per costruire pipeline di preprocessing leggibili e riutilizzabili.  
Il dataset resta quello del **marketing bancario**, ma l'attenzione si sposta dal *cosa* modelliamo a *come* lo facciamo in modo strutturato.

<div style="background-color:#f5f5f5; padding:10px; border-radius:4px">
<b>Nota importante</b><br>
Un modello addestrato senza tracciamento e senza pipeline resta un prototipo.  
Un modello inserito in una pipeline, tracciato con MLflow e salvato in un formato standard comincia ad assomigliare a qualcosa che può vivere fuori dal notebook.
</div>



<a id="setup"></a>
## Setup: librerie, esperimento MLflow e dati grezzi

Per prima cosa importiamo le librerie, configuriamo un esperimento MLflow dedicato a questo modulo e carichiamo i dati grezzi del marketing bancario dalla libreria `ucimlrepo`.

L'esperimento MLflow ci permetterà di raccogliere in un unico contenitore tutte le esecuzioni di training che facciamo qui.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Caricamento dataset Bank Marketing dalla UCI
bank_marketing = fetch_ucirepo(id=222)

X_raw = bank_marketing.data.features.copy()
y_raw = bank_marketing.data.targets.copy()
# Sample di 10kcrighe per velocizzare l'esecuzione
X_raw, _, y_raw, _ = train_test_split(X_raw, y_raw, train_size=10000, random_state=42, stratify=y_raw)

print("Shape originale X:", X_raw.shape)
print("Shape originale y:", y_raw.shape)
print("Prime colonne:", X_raw.columns.tolist()[:10])

display(X_raw.head())
display(y_raw.head())


Shape originale X: (10000, 16)
Shape originale y: (10000, 1)
Prime colonne: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day_of_week']


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
14570,40,entrepreneur,single,secondary,no,5,no,yes,cellular,15,jul,117,3,-1,0,NaN
31656,36,self-employed,married,secondary,no,1184,yes,no,cellular,6,apr,1063,1,-1,0,NaN
33133,32,services,married,secondary,no,197,yes,no,cellular,20,apr,421,4,-1,0,NaN
23082,32,technician,single,secondary,no,-5,no,yes,cellular,26,aug,300,4,-1,0,NaN
40849,64,management,married,secondary,no,1646,no,no,cellular,12,aug,68,3,-1,0,NaN


,y
14570,no
31656,yes
33133,no
23082,no
40849,no



<a id="prep-split"></a>
## Preparazione del dataset e train/test split

In questa sezione prepariamo i dati usando l'approccio **modulare con pipeline** visto in T2 e T3.

**Vantaggi dell'approccio pipeline:**
- **Coerenza**: stesso preprocessing in train e test
- **Riproducibilità**: l'intera pipeline è serializzabile con MLflow
- **Modularità**: possiamo testare diversi modelli con lo stesso preprocessor
- **Manutenibilità**: modifiche al preprocessing in un unico punto

**Passi:**
1. Mappiamo il target `y` da `yes`/`no` a 1/0
2. Rimuoviamo `duration` (informazione post-chiamata non disponibile a priori)
3. Selezioniamo le feature e le categorizziamo (binarie, ordinali, nominali, numeriche)
4. Creiamo train/test split
5. Costruiamo un `ColumnTransformer` modulare con transformer dedicati per ogni tipo


In [ ]:

from sklearn import set_config
set_config(transform_output="pandas")

# Mappatura del target y: yes/no -> 1/0
y = y_raw.iloc[:, 0].map({'yes': 1, 'no': 0})

# Rimuoviamo duration (informazione post-chiamata)
if "duration" in X_raw.columns:
    X_raw = X_raw.drop(columns=["duration"])
    print("✓ Colonna 'duration' rimossa")

# Definiamo le colonne da usare (approccio modulare come in T2)
binary_cols = ['default', 'housing', 'loan']  # yes/no
ordinal_cols = ['education', 'month']  # con ordine
nominal_cols = ['job', 'marital', 'contact', 'poutcome']  # categoriche senza ordine
numeric_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

# Creiamo df_model con solo le colonne selezionate
all_cols = binary_cols + ordinal_cols + nominal_cols + numeric_cols
df_model = X_raw[all_cols].copy()

X = df_model

print(f'✓ Shape X: {X.shape}')
print(f'✓ Shape y: {y.shape}')
print(f'✓ Feature binarie: {len(binary_cols)}')
print(f'✓ Feature ordinali: {len(ordinal_cols)}')
print(f'✓ Feature nominali: {len(nominal_cols)}')
print(f'✓ Feature numeriche: {len(numeric_cols)}')

# Train/test split con stratificazione
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)



print(f'\n✓ Shape X_train: {X_train.shape}')print(f'✓ Shape X_test: {X_test.shape}')

Colonna target: y
Shape iniziale df: (10000, 17)
Colonna 'duration' rimossa dalle feature.
Shape X: (10000, 15)
Shape y: (10000,)
Shape X_train: (8000, 15)
Shape X_test : (2000, 15)



<a id="pipeline-feature-engine"></a>
## Pipeline di preprocessing modulare con ColumnTransformer

Per rendere il flusso ripetibile e riproducibile, costruiamo un **ColumnTransformer** che applica trasformazioni diverse a gruppi di feature:

**1. Binary encoder** (binarie yes/no):
- Imputation con moda
- OrdinalEncoder (no=0, yes=1)

**2. Ordinal encoder** (ordinali con ordine semantico):
- Imputation con moda
- OrdinalEncoder con categorie ordinate (es. education: primary < secondary < tertiary)

**3. Nominal encoder** (categoriche nominali):
- Imputation con moda
- OneHotEncoder con drop_first=True

**4. Numeric scaler** (numeriche continue):
- Imputation con mediana
- StandardScaler

Questo approccio modulare permette di:
- Testare facilmente diversi modelli con lo stesso preprocessing
- Serializzare l'intera pipeline con MLflow
- Mantenere coerenza tra train e test

**Riferimenti:**
- scikit-learn ColumnTransformer: https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
- scikit-learn Pipeline: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html  
- MeanMedianImputer: https://feature-engine.trainindata.com/en/latest/imputation/MeanMedianImputer.html  
- OneHotEncoder: https://feature-engine.trainindata.com/en/latest/encoding/OneHotEncoder.html  
- SklearnTransformerWrapper: https://feature-engine.trainindata.com/en/latest/wrappers/SklearnTransformerWrapper.html

<div style="background-color:#ffe9e9; padding:10px; border-radius:4px">
<b>Attenzione</b><br>
Tutta la logica di preprocessing deve stare nella pipeline. In questo modo lo stesso oggetto verrà usato sia in addestramento sia in inferenza, riducendo il rischio di discrepanze tra notebook e produzione.
</div>


In [ ]:
# =============================
# PREPROCESSOR MODULARE (stile T2)
# =============================

# 1. Transformer per binarie (yes/no → 0/1)
binary_encoder = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['no', 'yes']] * len(binary_cols),
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# 2. Transformer per ordinali (con ordine semantico)
education_order = ['unknown', 'primary', 'secondary', 'tertiary']
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

ordinal_encoder = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[education_order, month_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# 3. Transformer per nominali (one-hot encoding)
nominal_encoder = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        drop='first',
        sparse_output=False,
        handle_unknown='ignore'
    ))
])

# 4. Transformer per numeriche (impute + scale)
numeric_scaler = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 5. Combinazione con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_encoder, binary_cols),
        ('ordinal', ordinal_encoder, ordinal_cols),
        ('nominal', nominal_encoder, nominal_cols),
        ('numeric', numeric_scaler, numeric_cols)
    ],
    verbose_feature_names_out=False
)

print("✓ Preprocessor creato con 4 transformer modulari")
print(f"  - Binary encoder: {len(binary_cols)} colonne")
print(f"  - Ordinal encoder: {len(ordinal_cols)} colonne")
print(f"  - Nominal encoder: {len(nominal_cols)} colonne")
print(f"  - Numeric scaler: {len(numeric_cols)} colonne")


,age,education,default,balance,housing,loan,day_of_week,campaign,pdays,previous,...,month_aug,month_may,month_jul,month_feb,month_mar,month_oct,month_sep,poutcome_Missing,poutcome_success,poutcome_failure
24445,0.266667,1,1,0.065283,0,1,0.076923,-0.5,0.0,0.0,...,0,0,0,0,0,0,0,1,0,0
26768,1.200000,3,1,2.688679,1,1,0.307692,-0.5,0.0,0.0,...,0,0,0,0,0,0,0,1,0,0
42744,0.866667,1,1,0.409434,0,1,0.846154,-0.5,93.0,2.0,...,0,0,0,0,0,0,0,0,1,0
8073,-0.600000,1,1,-0.067547,0,0,-1.076923,-0.5,0.0,0.0,...,0,0,0,0,0,0,0,1,0,0
31626,-0.400000,2,1,-0.243396,0,1,-0.769231,0.0,223.0,17.0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
# ============================
# PIPELINE MODULARI PER DIVERSI MODELLI
# ============================

def make_pipeline(model, model_name):
    """Factory function che crea una pipeline completa: preprocessor + modello.
    
    Questo approccio modulare permette di:
    - Testare facilmente diversi modelli con lo stesso preprocessing
    - Serializzare l'intera pipeline (preprocessing + modello) con MLflow
    - Garantire la riproducibilità in produzione
    """
    return Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

# ============================
# DEFINIZIONE DEI MODELLI DA CONFRONTARE
# ============================

models = {
    'random_forest': make_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),
        'RandomForest'
    ),
    
    'gradient_boosting': make_pipeline(
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            min_samples_split=20,
            min_samples_leaf=10,
            subsample=0.8,
            random_state=42
        ),
        'GradientBoosting'
    ),
    
    'logistic_regression': make_pipeline(
        LogisticRegression(
            penalty='l2',

            C=0.1,

            solver='liblinear',    print(f"  - {name}")

            class_weight='balanced',for name in models.keys():

            random_state=42,print(f"✓ Creati {len(models)} modelli con pipeline dedicate:")

            max_iter=1000

        ),}

        'LogisticRegression'    )

    ),        'SVM'

            ),

    'decision_tree': make_pipeline(            random_state=42

        DecisionTreeClassifier(            probability=True,  # abilita predict_proba

            max_depth=8,            class_weight='balanced',

            min_samples_split=50,            gamma='scale',

            min_samples_leaf=20,            C=1.0,

            class_weight='balanced',            kernel='rbf',

            random_state=42        SVC(

        ),    'svm': make_pipeline(

        'DecisionTree'    
    ),

<a id="confronto-modelli"></a>
## Addestramento e confronto di più modelli con MLflow

Ora addestriamo i 5 modelli definiti, tracciando **ogni run con MLflow** in modo avanzato:

**Cosa logghiamo per ogni modello:**
1. **Metriche standard**: accuracy, precision, recall, F1, ROC AUC
2. **Tag personalizzati**: dataset, problem_type, model_type, feature_count
3. **Parametri del modello**: estratti automaticamente da autolog
4. **Pipeline completa**: preprocessing + modello serializzati insieme
5. **Artefatti di explainability**:
   - **PFI (Permutation Feature Importance)**: grafico delle feature più importanti
   - **SHAP**: summary plot che mostra l'impatto di ogni feature sulle predizioni

**Confronto tra modelli:**
- **RandomForest**: ensemble di alberi, robusto ma può fare overfitting
- **GradientBoosting**: boosting iterativo, spesso ottima performance
- **LogisticRegression**: lineare, interpretabile, veloce
- **DecisionTree**: singolo albero, interpretabile ma instabile
- **SVM**: kernel RBF, potente ma costoso computazionalmente

L'uso di MLflow ci permette di confrontare facilmente tutti i risultati nella UI.

In [ ]:
# ============================
# CONFIGURAZIONE MLflow
# ============================
import os
from pathlib import Path
import mlflow
import mlflow.sklearn

# Configurazione tracking (salva nella root del progetto)
root_dir = os.path.abspath("../")
mlflow.set_tracking_uri(f"file:{os.path.join(root_dir, 'mlruns')}")
mlflow.set_experiment("bank_marketing_modulo4_mlops")
mlflow.sklearn.autolog(log_models=False)  # disabilitiamo autolog models, li logghiamo manualmente

print(f"✓ MLflow configurato:")
print(f"  Tracking URI: {root_dir}/mlruns")
print(f"  Esperimento: bank_marketing_modulo4_mlops")

[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.1s
[Pipeline] ............. (step 2 of 2) Processing model, total=   0.3s


,steps,"[('preprocess', ...), ('model', ...)]"
,memory,None
,verbose,True
,steps,"[('median_num', ...), ('impute_cat', ...), ...]"
,memory,None
,verbose,False
,imputation_method,'median'
,variables,"['age', 'balance', ...]"
,imputation_method,'missing'
,fill_value,'Missing'
,variables,"['job', 'marital', ...]"


In [ ]:
# ============================
# FUNZIONE DI TRAINING AVANZATA CON MLflow
# ============================

import tempfile
import shap
from sklearn.inspection import permutation_importance

def train_and_log_model(model_name, pipeline, X_train, y_train, X_test, y_test):
    """
    Addestra un modello e logga tutto su MLflow:
    - Metriche di performance
    - Tag personalizzati
    - Pipeline completa
    - Artefatti di explainability (PFI + SHAP)
    """
    
    with mlflow.start_run(run_name=model_name):
        print(f"\n{'='*60}")
        print(f"Training: {model_name}")
        print(f"{'='*60}")
        
        # 1. TRAINING
        pipeline.fit(X_train, y_train)
        print("✓ Modello addestrato")
        
        # 2. PREDIZIONI
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        
        # 3. METRICHE
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_proba)
        
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_precision", prec)
        mlflow.log_metric("test_recall", rec)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_roc_auc", auc)
        
        print(f"✓ Metriche - ACC: {acc:.3f} | PREC: {prec:.3f} | REC: {rec:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
        
        # 4. TAG PERSONALIZZATI
        mlflow.set_tag("dataset", "bank_marketing")
        mlflow.set_tag("problem_type", "binary_classification")
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("n_features", X_train.shape[1])
        mlflow.set_tag("n_train_samples", X_train.shape[0])
        mlflow.set_tag("class_balance", f"{y_train.value_counts()[1]}/{y_train.value_counts()[0]}")
        
        # 5. LOG PIPELINE COMPLETA
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            input_example=X_test.head(3)
        )
        print("✓ Pipeline completa loggata")
        
        # 6. PERMUTATION FEATURE IMPORTANCE (PFI)
        print("  Calcolando PFI...", end=" ")
        pfi_result = permutation_importance(
            pipeline, X_test, y_test,
            n_repeats=10,
            random_state=42,
            n_jobs=-1
        )
        
        # Plot PFI
        importances_df = pd.DataFrame({
            'feature': X_test.columns,
            'importance': pfi_result.importances_mean,
            'std': pfi_result.importances_std
        }).sort_values('importance', ascending=False).head(15)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(importances_df['feature'][::-1], importances_df['importance'][::-1])
        ax.set_xlabel('Importance (drop in AUC)')
        ax.set_title(f'Permutation Feature Importance - {model_name}')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.png', delete=False) as f:
            fig.savefig(f.name, dpi=100, bbox_inches='tight')
            mlflow.log_artifact(f.name, artifact_path='explainability')
            os.unlink(f.name)
        plt.close(fig)
        print("✓")
        
        # 7. SHAP (solo per modelli tree-based)
        model_final = pipeline.named_steps['model']
        if hasattr(model_final, 'estimators_') or hasattr(model_final, 'tree_'):
            print("  Calcolando SHAP...", end=" ")
            
            # Preprocessa un sample del test set
            X_test_sample = X_test.sample(min(500, len(X_test)), random_state=42)
            X_test_transformed = pipeline.named_steps['preprocess'].transform(X_test_sample)
            
            # TreeExplainer
            explainer = shap.TreeExplainer(model_final)
            shap_values = explainer.shap_values(X_test_transformed)
            
            # Per binary classification, prendiamo i valori della classe positiva
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
            
            # Summary plot
            fig, ax = plt.subplots(figsize=(10, 8))
            shap.summary_plot(
                shap_values, 
                X_test_transformed,
                show=False,
                max_display=15
            )
            plt.title(f'SHAP Summary Plot - {model_name}')
            plt.tight_layout()
            
            with tempfile.NamedTemporaryFile(mode='w', suffix='.png', delete=False) as f:
                plt.savefig(f.name, dpi=100, bbox_inches='tight')
                mlflow.log_artifact(f.name, artifact_path='explainability')
                os.unlink(f.name)
            plt.close(fig)
            print("✓")
        else:
            print("  SHAP non disponibile per questo tipo di modello (solo tree-based)")
        
        print(f"✓ Run completata per {model_name}\n")
        return acc, prec, rec, f1, auc

print("✓ Funzione train_and_log_model definita")

month          0.075688
job            0.064150
balance        0.061413
age            0.053912
day_of_week    0.053175
housing        0.042650
campaign       0.040887
contact        0.040775
poutcome       0.031487
marital        0.030812
dtype: float64

In [ ]:
# ============================
# TRAINING DI TUTTI I MODELLI
# ============================

results = {}

for model_name, pipeline in models.items():
    metrics = train_and_log_model(
        model_name, 
        pipeline, 
        X_train, y_train, 
        X_test, y_test
    )
    results[model_name] = metrics

# Creiamo un DataFrame per confrontare i risultati
results_df = pd.DataFrame(results, index=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']).T
results_df = results_df.sort_values('roc_auc', ascending=False)

print("\n" + "="*80)
print("RISULTATI FINALI - Confronto modelli")
print("="*80)
display(results_df.round(4))

month          0.075688
job            0.064150
balance        0.061413
age            0.053912
day_of_week    0.053175
housing        0.042650
campaign       0.040887
contact        0.040775
poutcome       0.031487
marital        0.030812
dtype: float64


<a id="mlflow-ui"></a>
## Lettura degli esperimenti dalla UI di MLflow

Gli esperimenti registrati da questo notebook possono essere ispezionati in modo più comodo tramite l'interfaccia web di MLflow.

Dal terminale, nella cartella del progetto, si può eseguire:

```bash
mlflow ui --port 5000
```

e aprire il browser all'indirizzo:

```text
http://127.0.0.1:5000
```

Dentro la UI troverai:

- l'esperimento `bank_marketing_modulo4_mlops`;
- le run per ciascun modello, con le metriche che abbiamo loggato sul test set;
- i parametri dei modelli, l'ambiente software e il modello salvato in formato MLflow.

Questa schermata diventa il punto di riferimento per:

- confrontare i risultati dei diversi tentativi;
- ritrovare a distanza di tempo il modello effettivamente scelto;
- capire quali impostazioni hanno portato a una certa performance.


<a id="esercizio"></a>
## Esercizio guidato

<div style="background-color:#f0f7ff; padding:15px; border-radius:6px; border-left: 4px solid #2196F3;">
<b>🎯 Esercizio: Ottimizzazione degli iperparametri</b><br><br>

L'obiettivo è esplorare come diversi iperparametri influenzano le performance e come MLflow facilita il confronto.

<b>Task 1 - Random Forest</b>
<ol>
<li>Nel dizionario <code>models</code>, crea una variante <code>random_forest_light</code> con:
   - <code>n_estimators=50</code>
   - <code>max_depth=5</code>
   - <code>min_samples_split=50</code>
</li>
<li>Addestra il nuovo modello e confronta nella UI MLflow:
   - Accuracy e AUC cambiano?
   - Il tempo di training è ridotto?
   - Guarda i plot PFI: le feature importanti sono le stesse?
</li>
</ol>

<b>Task 2 - Logistic Regression</b>
<ol>
<li>Crea due varianti modificando il parametro <code>C</code>:
   - <code>logistic_c001</code> con C=0.01 (forte regolarizzazione)
   - <code>logistic_c10</code> con C=10 (debole regolarizzazione)
</li>
<li>Confronta precision vs recall: quale variante è più conservativa?</li>
</ol>

<b>Task 3 - Gradient Boosting</b>
<ol>
<li>Modifica <code>learning_rate</code> e <code>n_estimators</code>:
   - learning_rate=0.01 con n_estimators=200
   - learning_rate=0.3 con n_estimators=50
</li>
<li>Quale combinazione offre il miglior compromesso performance/tempo?</li>
</ol>

<b>🤔 Domanda di riflessione:</b><br>
In una campagna marketing, preferisci un modello con alta precision (poche chiamate sbagliate) o alta recall (catturi più clienti interessati)? Come influenza questa scelta la tua scelta del modello?

<b>💡 Suggerimento:</b> Usa i tag MLflow per annotare le tue osservazioni direttamente nelle run!
</div>

<a id="modello-inferenza"></a>
## Dal modello tracciato all'inferenza riutilizzabile

Uno dei vantaggi principali di MLflow è la possibilità di ricaricare un modello salvato e usarlo per l'inferenza in produzione.

**Cosa abbiamo loggato:**

✓ **Pipeline completa** (preprocessing + modello):
- Imputation
- Encoding (binary, ordinal, nominal)
- Scaling
- Modello finale

✓ **Artefatti di explainability**:
- Permutation Feature Importance (PFI)
- SHAP summary plots (per modelli tree-based)

✓ **Metriche e tag** per confrontare facilmente le run

**Come ricaricare il miglior modello:**

1. Cerchiamo la run con miglior AUC usando `MlflowClient`
2. Carichiamo la pipeline completa
3. Facciamo predizioni su nuovi dati (la pipeline gestisce automaticamente il preprocessing!)

**Nota importante:** La pipeline loggata contiene TUTTO il necessario per fare inferenza. Non serve più nessun preprocessing manuale!

In [ ]:
# ============================
# RECUPERO AUTOMATICO DELLA MIGLIORE RUN
# ============================

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Ottieni l'experiment_id
experiment = client.get_experiment_by_name("bank_marketing_modulo4_mlops")
if experiment is None:
    print("❌ Esperimento non trovato! Esegui prima le celle di training.")
else:
    experiment_id = experiment.experiment_id
    
    # Cerca tutte le run ordinate per AUC (migliore prima)
    runs = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string="metrics.test_roc_auc > 0",
        order_by=["metrics.test_roc_auc DESC"],
        max_results=5
    )
    
    if runs:
        print("\n" + "="*80)
        print("TOP 5 MODELLI PER ROC AUC")
        print("="*80)
        
        for i, run in enumerate(runs, 1):
            model_name = run.data.tags.get('model_type', 'N/A')
            auc = run.data.metrics.get("test_roc_auc", 0)
            acc = run.data.metrics.get("test_accuracy", 0)
            f1 = run.data.metrics.get("test_f1", 0)
            print(f"{i}. {model_name:25s} | AUC: {auc:.4f} | ACC: {acc:.4f} | F1: {f1:.4f}")
        
        # Selezioniamo il migliore
        best_run = runs[0]
        best_run_id = best_run.info.run_id
        best_model_name = best_run.data.tags.get('model_type', 'N/A')
        best_auc = best_run.data.metrics.get("test_roc_auc", 0)
        
        print("\n" + "="*80)
        print(f"✓ MIGLIORE MODELLO SELEZIONATO: {best_model_name}")
        print(f"  Run ID: {best_run_id}")
        print(f"  ROC AUC: {best_auc:.4f}")
        print("="*80)
        
        # ============================
        # CARICAMENTO DELLA PIPELINE
        # ============================
        
        model_uri = f"runs:/{best_run_id}/model"
        
        try:
            loaded_pipeline = mlflow.sklearn.load_model(model_uri)
            print(f"\n✓ Pipeline caricata con successo!")
            print(f"  URI: {model_uri}")
            
            # ============================
            # TEST SU NUOVI DATI
            # ============================
            
            # Prendiamo un campione del test set
            sample = X_test.head(10).copy()
            
            # La pipeline gestisce automaticamente il preprocessing!
            predictions = loaded_pipeline.predict(sample)
            probabilities = loaded_pipeline.predict_proba(sample)[:, 1]
            
            # Creiamo un DataFrame con i risultati
            results = pd.DataFrame({
                'actual': y_test.head(10).values,
                'predicted': predictions,
                'proba_conversion': probabilities.round(3)
            })
            
            # Aggiungiamo qualche feature importante per interpretabilit\u00e0
            results['age'] = sample['age'].values
            results['balance'] = sample['balance'].values
            results['campaign'] = sample['campaign'].values
            
            print("\n" + "="*80)
            print("PREDIZIONI SU NUOVI DATI (campione di 10 clienti)")
            print("="*80)
            display(results)
            
            # Accuracy sul campione
            sample_acc = accuracy_score(results['actual'], results['predicted'])
            print(f"\n✓ Accuracy sul campione: {sample_acc:.2%}")
            
        except Exception as e:
            print(f"\n❌ Errore nel caricamento: {e}")
    else:
        print("❌ Nessuna run trovata con metriche valide!")

✓ Migliore run trovata:
  Run ID: 029ae25c56f5402b96015d5c4a446f06
  Model: Random_Forest
  AUC: 0.7767



✓ Pipeline caricata con successo!
  URI: runs:/029ae25c56f5402b96015d5c4a446f06/pipeline

✓ Previsioni sulla pipeline caricata:


,y_actual,y_pred,p_conversion
0,1,0,0.010
1,0,0,0.110
2,0,0,0.010
3,0,0,0.270
4,1,0,0.125
5,0,0,0.015
6,1,0,0.405
7,0,0,0.005
8,0,0,0.045
9,0,0,0.015


## 🎓 Recap: Cosa abbiamo imparato in questo modulo

**1. Pipeline modulari scikit-learn:**
- `ColumnTransformer` per gestire tipi di feature diversi
- Transformer dedicati: binary, ordinal, nominal, numeric
- Pipeline complete serializzabili (preprocessing + modello)

**2. MLflow per MLOps:**
- ✅ **Autolog**: parametri e metriche automatiche
- ✅ **Metriche custom**: test_accuracy, test_precision, test_recall, test_f1, test_roc_auc
- ✅ **Tag personalizzati**: dataset, problem_type, model_type, n_features, class_balance
- ✅ **Artefatti**: pipeline completa + explainability (PFI, SHAP)

**3. Model explainability:**
- **PFI (Permutation Feature Importance)**: quali feature sono più importanti per le predizioni
- **SHAP**: come ogni feature contribuisce alle singole predizioni (solo tree-based)

**4. Confronto di modelli:**
- RandomForest, GradientBoosting, LogisticRegression, DecisionTree, SVM
- Confronto sistematico tramite MLflow UI
- Ricaricamento automatico del modello migliore

**5. Produzione:**
- L'oggetto caricato da MLflow contiene la pipeline completa
- Basta chiamare `.predict()` su dati raw (preprocessing automatico!)
- Riproducibilità garantita: versioni librerie, ambiente, parametri

---

**🚀 Prossimi passi:**
- Esplora la MLflow UI (`mlflow ui --port 5000`)
- Confronta i plot PFI e SHAP dei diversi modelli
- Prova l'esercizio guidato per ottimizzare gli iperparametri
- Registra il modello migliore nel Model Registry per deployment